# 🏥 MedViT-Lite: Inference & Clinical Explainability Demo

This notebook demonstrates how to:
1. Load the trained **MedViT-Lite** model.
2. Run inference on Chest X-ray images from ChestMNIST.
3. Compute **Monte Carlo Dropout Uncertainty** for all 14 thoracic pathologies.
4. Generate **Grad-CAM++ visual heatmaps** showing what regions the model focused on.

In [ ]:
import os
import sys
import yaml
import torch
import numpy as np
import matplotlib.pyplot as plt

# Ensure project root is in path
sys.path.insert(0, os.path.abspath(".."))

from models.medvit_lite import MedViTLite
from data.datasets.chest_mnist import PATHOLOGY_NAMES, build_dataloaders
from training.trainer import Trainer
from explainability.gradcam import GradCAMPlusPlus

print("✅ Environment & modules loaded successfully.")

## 1. Load Configuration & Model

In [ ]:
with open("../configs/base.yaml", "r") as f:
    config = yaml.safe_load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_cfg = config["model"]

model = MedViTLite(
    num_classes=config["data"]["num_classes"],
    image_size=config["data"]["image_size"],
    patch_size=model_cfg["patch_size"],
    embed_dim=model_cfg["embed_dim"],
    local_depth=4,
    global_depth=2,
    num_heads=model_cfg["attention"]["num_heads"],
    use_dps=model_cfg["sparsifier"]["enabled"],
    keep_ratio=model_cfg["sparsifier"]["keep_ratio"],
    use_sfc=model_cfg["frame_cache"]["enabled"],
    mc_samples=model_cfg["head"]["mc_samples"],
).to(device)

ckpt_path = "../checkpoints/best_medvit_lite.pth"
if os.path.exists(ckpt_path):
    Trainer.load_checkpoint(ckpt_path, model, str(device))
    print(f"✅ Loaded checkpoint: {ckpt_path}")
else:
    print("ℹ️ Checkpoint not found locally. Running with initialized weights.")

model.eval()
print(f"Model parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")

## 2. Load Sample Chest X-Ray from Test Set

In [ ]:
_, _, test_loader = build_dataloaders(config["data"], root="../data/raw", dry_run=False)
sample_idx = 0
raw_img, ground_truth = test_loader.dataset[sample_idx]

img_tensor = Trainer.preprocess_batch(raw_img.unsqueeze(0), device=device, is_train=False)

print(f"Input shape: {img_tensor.shape}")
print("Ground truth positive findings:")
for idx, val in enumerate(ground_truth):
    if val > 0.5:
        print(f"  - {PATHOLOGY_NAMES[idx]}")

## 3. Predict Multi-Label Pathologies with Epistemic Uncertainty

In [ ]:
with torch.no_grad():
    mean_probs, uncertainty = model.predict_with_uncertainty(img_tensor, num_samples=15)

probs = mean_probs.squeeze().cpu().numpy()
unc = uncertainty.squeeze().cpu().numpy()

top_idx = int(np.argmax(probs))
print(f"Top predicted finding: {PATHOLOGY_NAMES[top_idx]} ({probs[top_idx]*100:.1f}% ± {unc[top_idx]*100:.1f}%)")

## 4. Visual Explainability (Grad-CAM++ Overlay)

In [ ]:
target_layer = model.attention.local_blocks[-1].norm1
gradcam = GradCAMPlusPlus(model, target_layer)
heatmap = gradcam.generate_heatmap(img_tensor, target_class=top_idx)

display_img = img_tensor[0].permute(1, 2, 0).detach().cpu().numpy()
display_img = (display_img - display_img.min()) / (display_img.max() - display_img.min() + 1e-6)
overlay = gradcam.overlay_heatmap(display_img, heatmap)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(display_img)
axes[0].set_title("Chest X-Ray Input", fontsize=12, fontweight="bold")
axes[0].axis("off")

axes[1].imshow(overlay)
axes[1].set_title(f"Grad-CAM++ Focus: {PATHOLOGY_NAMES[top_idx]}", fontsize=12, fontweight="bold", color="darkred")
axes[1].axis("off")

y_pos = np.arange(len(PATHOLOGY_NAMES))
axes[2].barh(y_pos, probs * 100, xerr=unc * 100, align="center", color="#3B82F6", alpha=0.8, capsize=3)
axes[2].set_yticks(y_pos)
axes[2].set_yticklabels(PATHOLOGY_NAMES, fontsize=9)
axes[2].invert_yaxis()
axes[2].set_xlabel("Probability (%) ± MC Uncertainty", fontsize=10)
axes[2].set_title("Multi-Label Predictions & Confidence", fontsize=12, fontweight="bold")
axes[2].set_xlim(0, 100)
axes[2].grid(axis="x", linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()
gradcam.remove_hooks()